Ce projet consite a crée un agent IA(personnal_chef ) qui :
* Prendre les **ingrédients qu’il vous reste dans la cuisine**.
* **Rechercher sur Internet** des recettes utilisant ces ingrédients[Tools].
* **Vous proposer les recettes trouvées**.
* Pouvoir **répondre à vos questions de suivi**, par exemple en gardant en mémoire les échanges précédents, c’est-à-dire qu’il possède une **« mémoire » des exécutions précédentes**.


In [1]:
from dotenv import load_dotenv
load_dotenv()



True

In [2]:
#Recherche internet 
from langchain.tools import tool
from typing import Dict, Any 
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool("web_search")
def web_search(query: str) ->str:
    """Search the web for information."""
    return tavily_client.search(query)


In [3]:
system_prompt = """
    Tu es un chef de cuisine. 
    Tu utilisera les ingrédients se trouvant sur la photo que j'ai upload, pour rechercher sur internet des menues qu'on peut faires avec ses ingrédients.
    Tu vas par la suite me proposer à base de tes recherche des menus.
    Tu gadera en mémoire courte(short-term) les échanges précédents.
    Tu ne repondra qu'au question en lien avec la cuisine car tu es un chef de cuisine."""
    

In [4]:
# Création de l'agent avec le short-term memory 
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
model = ChatOpenAI(model="gpt-4.1-mini")
agent = create_agent(
    model = model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

In [5]:
#Dit a lagent ce qui se trouve dans notre cuisine 
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?. please answer in french")]},
    config
)

print(response['messages'][-1].content)

Avec des restes de poulet et de riz, voici quelques idées de menus que vous pouvez préparer :

1. Riz sauté au poulet à l'asiatique : faites revenir des légumes (carottes, petits pois, cébettes), ajoutez des œufs brouillés, le poulet coupé en morceaux, puis mélangez avec le riz et un peu de sauce soja. Un plat rapide et savoureux.

2. Reste de volaille au riz et à la crème : faire revenir le poulet avec un peu de moutarde à l'estragon, puis ajouter de la crème fraîche et mélanger avec du riz basmati cuit.

3. Riz aux restes de poulet rôti : préparer un bouillon avec la carcasse du poulet pour parfumer le riz, puis y ajouter les morceaux de poulet et un peu de persil.

4. Poivrons farcis au riz et poulet : mélanger le riz et le poulet avec des légumes ou épices de votre choix, farcir des poivrons et cuire au four.

Voulez-vous que je vous détaille une recette en particulier ?


In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?. please answer in french', additional_kwargs={}, response_metadata={}, id='abe3159a-1c52-47fc-82fa-5eee8428e4d2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 156, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5fc4ff561d', 'id': 'chatcmpl-ESH0QVp3Bnj9QgUDrIOIJ5UVoiQPq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0dc96-b931-7571-841b-ebf9b81521ad-0', tool_calls=[{'name': 'web_search'

In [7]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

In [8]:
print(response["messages"][-1].content)

Bonjour ! Je suis là pour toutes vos questions liées à la cuisine. Souhaitez-vous des idées de recettes avec vos ingrédients ?
